# Distributed Sequencer Architecture Playground

This notebook exercises the same application code used by the CLI. It is intentionally a client of the package rather than the home of system logic.

In [ ]:
import importlib.util
import os
import platform
import sys

print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("cpu_count:", os.cpu_count())

torch_spec = importlib.util.find_spec("torch")
print("torch_installed:", torch_spec is not None)

if torch_spec is not None:
    import torch

    print("torch_version:", torch.__version__)
    print("cuda_available:", torch.cuda.is_available())
    print("cuda_device_count:", torch.cuda.device_count())

    for index in range(torch.cuda.device_count()):
        print(f"cuda_device_{index}:", torch.cuda.get_device_name(index))

    mps_available = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
    print("mps_available:", mps_available)

    if torch.cuda.is_available():
        selected_device = "cuda"
    elif mps_available:
        selected_device = "mps"
    else:
        selected_device = "cpu"

    print("selected_device:", selected_device)
else:
    print("selected_device:", "cpu")

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(repo_root / "src"))
repo_root

In [ ]:
from distributed_sequencer.application.composition import (
    CompositionEngine,
    DensityCritic,
    ProceduralCompositionModel,
    RegisterCritic,
)
from distributed_sequencer.domain.state import CompositionContext

engine = CompositionEngine(
    ProceduralCompositionModel(seed=42),
    (DensityCritic(), RegisterCritic()),
)
context = CompositionContext(role="bass", root_pitch=36, desired_density=1.0)
phrase = await engine.compose(context)
phrase

In [ ]:
from distributed_sequencer.application.variation import VariationEngine
from distributed_sequencer.domain.state import VariationPolicy

variation = VariationEngine(seed=123)
policy = VariationPolicy(timing_jitter_ticks=1, velocity_jitter=6, omission_probability=0.1)
await variation.vary(phrase, policy, generation=1)